## Used Car Price Prediction

### 1) What is the problem?
CarDekho is a website in India where people buy and sell used cars.

When someone wants to sell their car, the hardest question is: **"What price should I ask for?"**
- If the price is too high, nobody buys the car.
- If the price is too low, the seller loses money.

**Our goal:** Predict the **selling price** of a used car from its details (like age, km driven, fuel type). This helps sellers pick a fair price.

This is a **regression** problem, because we are predicting a number (price), not a category.

### 2) About the data
- Source: Collected from the CarDekho website
- Size: **15411 rows** and **13 columns**

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [2]:
# The first column in the file is just row numbers, so we use it as the index
df = pd.read_csv("cardekho.csv", index_col=0)
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## Data Cleaning
This dataset was already cleaned before ("imputated" in the file name means missing values were already filled).

So here we will only:
1. **Check missing values** - just to confirm there are none
2. **Remove unused columns** - columns that repeat the same information

In [3]:
# Count of missing values in each column
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

### Remove unused columns
Look carefully at these three columns: `car_name`, `brand` and `model`.

- `car_name` is just `brand` + `model` joined together (Maruti + Alto = Maruti Alto).
- The `model` already tells us the brand. An Alto is always a Maruti, a City is always a Honda.

So in real life, if you know the car model, you already know everything the other two columns tell you.
That's why we keep only `model` and remove `car_name` and `brand`.

In [4]:
df = df.drop(columns=['car_name', 'brand'])
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## Types of Columns
Let's see what kind of columns we have.

In [5]:
# Columns that store numbers
num_features = df.select_dtypes(include='number').columns
print(f'Num of Numerical Features : {len(num_features)}')

Num of Numerical Features : 7


In [6]:
# Columns that store text
cat_features = df.select_dtypes(exclude='number').columns
print(f'Num of Categorical Features : {len(cat_features)}')

Num of Categorical Features : 4


In [7]:
# Discrete = numeric columns with only a few unique values (like number of seats: 4, 5, 7...)
# Here, a column with 25 or fewer unique values is treated as discrete
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print(f'Num of Discrete Features : {len(discrete_features)}')

Num of Discrete Features : 2


In [8]:
# Continuous = numeric columns with many different values (like km driven or selling price)
# Here, every numeric column that is not discrete is treated as continuous
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print(f'Num of Continuous Features : {len(continuous_features)}')

Num of Continuous Features : 5


## Input and Output
- **X (input):** all the car details we use to make the prediction
- **y (output):** `selling_price`, the thing we want to predict

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['selling_price'])
y = df['selling_price']
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


## Converting Text Columns to Numbers
Models only understand numbers, so we need to convert the text columns.

- `seller_type`, `fuel_type` and `transmission_type` have only a few values, so we use **One Hot Encoding** (one new 0/1 column for each value).
- `model` has **120** different car models. One Hot Encoding would add 120 new columns, which is too many. So we use **Label Encoding** instead (each car model gets its own number).

In [10]:
# Number of different values in each text column
for feature in cat_features:
    print(f'{feature} : {df[feature].nunique()} unique values')

model : 120 unique values
seller_type : 3 unique values
fuel_type : 5 unique values
transmission_type : 2 unique values


In [11]:
from sklearn.preprocessing import LabelEncoder

# Give each car model its own number
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [12]:
# Split data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training rows : {len(X_train)}')
print(f'Testing rows : {len(X_test)}')

Training rows : 12328
Testing rows : 3083


In [13]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Text columns with few values, and all number columns
onehot_columns = ['seller_type', 'fuel_type', 'transmission_type']
num_features = X.select_dtypes(include='number').columns

# Text columns -> convert to numbers (OneHotEncoder)
# Number columns -> bring to the same scale (StandardScaler)
preprocessor = ColumnTransformer(
    [
        # drop='first' removes one extra column per category, as it can be guessed from the others
        # Example: transmission_type has Automatic and Manual -> we keep only transmission_type_Manual
        #   transmission_type_Manual = 1 means Manual, transmission_type_Manual = 0 means Automatic
        ("OneHotEncoder", OneHotEncoder(drop='first'), onehot_columns),
        # StandardScaler is not needed for decision trees (or random forest), as they only compare values, scale does not matter
        # We still use it because Linear Regression and KNN are also trained below, and they need scaled data
        ("StandardScaler", StandardScaler(), num_features)
    ]
)

In [14]:
# fit_transform on training data: learn the values (like mean) and apply the changes
X_train = preprocessor.fit_transform(X_train)

In [15]:
# transform on test data: use the values learned from training data (no fit here)
# Test data should stay unseen, so we don't learn anything from it
X_test = preprocessor.transform(X_test)

In [16]:
pd.DataFrame(X_train).head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.255968,0.323969,0.349100,-2.050819,1.756765,2.681685,-0.403824
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.789199,-1.337798,-1.069394,0.985661,-0.547081,-0.382744,-0.403824
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.242618,-1.337798,-1.163564,-0.177042,0.893542,3.296910,-0.403824
3,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.022962,0.323969,0.178369,-0.465315,0.024564,0.396229,-0.403824
4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.503081,1.321030,0.585469,0.149668,-0.550917,-0.502047,-0.403824


## Model Training
We will train 7 different models and compare how well each one predicts the price.

In [17]:
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [18]:
# Models we want to compare
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor()
}

# Prints all the scores by comparing real prices with predicted prices
def print_scores(y_true, y_pred):
    # MAE: on average, how far the predicted price is from the real price
    print(f"- MAE      : {mean_absolute_error(y_true, y_pred):.2f}")

    # RMSE: similar to MAE, but big mistakes are punished more
    print(f"- RMSE     : {np.sqrt(mean_squared_error(y_true, y_pred)):.2f}")

    # R2 score: how well the model explains the price
    # 1.0 = perfect, 0 = no better than just guessing the average price
    print(f"- R2 score : {r2_score(y_true, y_pred):.4f}")

# Train each model and check how well it does
for name, model in models.items():
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    print(name)
    print("Training set:")
    print_scores(y_train, y_train_pred)

    print("Test set:")
    print_scores(y_test, y_test_pred)

    print("=" * 35)
    print()

Linear Regression
Training set:
- MAE      : 268101.61
- RMSE     : 553855.67
- R2 score : 0.6218
Test set:
- MAE      : 279618.58
- RMSE     : 502543.59
- R2 score : 0.6645

Lasso
Training set:
- MAE      : 268099.23
- RMSE     : 553855.67
- R2 score : 0.6218
Test set:
- MAE      : 279614.76
- RMSE     : 502542.74
- R2 score : 0.6645

Ridge
Training set:
- MAE      : 268060.01
- RMSE     : 553856.32
- R2 score : 0.6218
Test set:
- MAE      : 279557.45
- RMSE     : 502534.47
- R2 score : 0.6645

K-Neighbors Regressor
Training set:
- MAE      : 92495.94
- RMSE     : 321491.14
- R2 score : 0.8726
Test set:
- MAE      : 112934.07
- RMSE     : 257765.16
- R2 score : 0.9117

Decision Tree
Training set:
- MAE      : 5164.82
- RMSE     : 20797.24
- R2 score : 0.9995
Test set:
- MAE      : 125137.55
- RMSE     : 306888.29
- R2 score : 0.8749

Random Forest Regressor
Training set:
- MAE      : 39870.08
- RMSE     : 133980.32
- R2 score : 0.9779
Test set:
- MAE      : 101562.26
- RMSE     : 2252

## Hyperparameter Tuning
Now we take **KNN**, **Random Forest** and **AdaBoost**, and try different settings for them to find which settings work best.

In [19]:
# Settings we want to try for KNN
knn_params = {
    "n_neighbors": [2, 3, 10, 20, 40, 50]       # How many nearest cars to look at
}

# Settings we want to try for Random Forest
rf_params = {
    "max_depth": [5, 8, 10, 15, None],          # How deep each tree can grow (None = no limit)
    "max_features": [5, 7, 8, 1.0],             # How many columns each tree looks at while splitting (1.0 = all columns)
    "min_samples_split": [2, 8, 15, 20],        # Minimum rows needed to split a node
    "n_estimators": [100, 200, 500, 1000]       # Number of trees in the forest
}

# Settings we want to try for AdaBoost
#
# How AdaBoost works for regression:
#   It builds small trees one after another. After each tree, the cars whose price it got most wrong get a higher weight,
#   so the next tree focuses more on those hard cars. At the end, all trees are combined, and better trees get a bigger say.
#
# learning_rate:
#   Every tree's say is multiplied by the learning_rate.
#   - learning_rate = 1    -> each tree has its full say, so the model learns fast (but can overfit)
#   - learning_rate = 0.1  -> each tree has only 10% of its say, so the model learns slowly and carefully
#   A small learning_rate usually needs more trees (higher n_estimators) to reach the same result.
#
# loss:
#   After each tree, AdaBoost checks how wrong it was for every car.
#   First the mistake is scaled between 0 and 1: the car with the biggest mistake gets 1, a perfect guess gets 0.
#   Then "loss" decides how that 0-1 mistake is turned into a penalty:
#   - 'linear'      : penalty = mistake
#                     Example: mistake 0.1 -> 0.1, mistake 0.5 -> 0.5. Every mistake counts as it is.
#   - 'square'      : penalty = mistake * mistake
#                     Example: mistake 0.1 -> 0.01, mistake 0.5 -> 0.25.
#                     Small mistakes become tiny, so the next trees focus mostly on the cars with big mistakes.
#   - 'exponential' : penalty = 1 - e^(-mistake)
#                     Example: mistake 0.1 -> 0.10, mistake 0.5 -> 0.39, mistake 1 -> 0.63.
#                     Big mistakes are capped, so a few very wrong cars (outliers) can't take over the training.
#   The penalty decides two things: how much say the tree gets, and how much weight each car gets for the next tree.
ada_params = {
    "n_estimators": [50, 60, 70, 80, 90],               # Number of small trees built one after another
    "learning_rate": [0.001, 0.01, 0.1, 1],             # How much say each tree gets in the final result (lower = slower, careful learning)
    "loss": ["linear", "square", "exponential"]         # How a price mistake is turned into a penalty (see above)
}

In [20]:
# List of models to tune: (short name, model, settings to try)
randomcv_models = [
    ("KNN", KNeighborsRegressor(), knn_params),
    ("RF", RandomForestRegressor(), rf_params),
    ("ADA", AdaBoostRegressor(), ada_params)
]

In [21]:
from sklearn.model_selection import RandomizedSearchCV

# Store the best settings found for each model
model_param = {}

for name, model, params in randomcv_models:
    # RandomizedSearchCV tries random combinations of settings and keeps the best one
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=100,     # Try 100 random combinations
        cv=3,           # Check each combination 3 times on different parts of the data
        verbose=2,      # Show progress while running
        n_jobs=-1       # Use all CPU cores to run faster
    )
    search.fit(X_train, y_train)
    model_param[name] = search.best_params_

# Show the best settings for each model
for model_name in model_param:
    print(f"---------------- Best Params for {model_name} ----------------")
    print(model_param[model_name])

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 60 candidates, totalling 180 fits
---------------- Best Params for KNN ----------------
{'n_neighbors': 10}
---------------- Best Params for RF ----------------
{'n_estimators': 500, 'min_samples_split': 2, 'max_features': 7, 'max_depth': 15}
---------------- Best Params for ADA ----------------
{'n_estimators': 80, 'loss': 'exponential', 'learning_rate': 0.1}


## Retrain With Best Settings

In [22]:
# Train all three models again, this time using the best settings we found above
best_rf = model_param["RF"]
best_knn = model_param["KNN"]
best_ada = model_param["ADA"]

models = {
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=best_rf["n_estimators"],
        min_samples_split=best_rf["min_samples_split"],
        max_features=best_rf["max_features"],
        max_depth=best_rf["max_depth"]
    ),
    "K-Neighbors Regressor": KNeighborsRegressor(
        n_neighbors=best_knn["n_neighbors"]
    ),
    "AdaBoost Regressor": AdaBoostRegressor(
        n_estimators=best_ada["n_estimators"],
        learning_rate=best_ada["learning_rate"],
        loss=best_ada["loss"]
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    print(name)
    print("Training set:")
    print_scores(y_train, y_train_pred)

    print("Test set:")
    print_scores(y_test, y_test_pred)

    print("=" * 35)
    print()

Random Forest Regressor
Training set:
- MAE      : 54638.04
- RMSE     : 138496.99
- R2 score : 0.9763
Test set:
- MAE      : 97833.72
- RMSE     : 213503.43
- R2 score : 0.9394

K-Neighbors Regressor
Training set:
- MAE      : 104560.11
- RMSE     : 364484.80
- R2 score : 0.8362
Test set:
- MAE      : 118507.05
- RMSE     : 265043.07
- R2 score : 0.9067

AdaBoost Regressor
Training set:
- MAE      : 225848.03
- RMSE     : 362033.51
- R2 score : 0.8384
Test set:
- MAE      : 243898.40
- RMSE     : 395106.33
- R2 score : 0.7926

